# FX Triangular Arbitrage: Compile a Market Opportunity into a Sign

## What this demonstrates

**ZeroModel can compile identified FX quotes and a declared triangular-arbitrage calculation into a deterministic VPM where the strongest executable opportunity occupies a predictable location and can be recovered with its complete calculation and source trace.**

This is not statistical arbitrage. It uses the exact three-currency bid/ask relationship among EUR, USD, and GBP.

## Why it matters

The goal is not better arithmetic than ordinary Python or pandas. The goal is stable identities, explicit layout, source-to-view mapping, artifact provenance, mutation identity, replay, and a shared human/machine sign.

## Source and package mapping

- Source example: `examples/fx_triangular_arbitrage.py`
- Packages: `zeromodel.core`, `zeromodel.artifacts`
- No live broker API, no external data, no embeddings, no `zeromodel-search`.

## Application

Identified bid/ask quotes become deterministic cycle calculations, cost and freshness checks, a scored opportunity surface, a VPM, and a top-left `EXECUTE`, `SKIP`, or `REJECT` sign.

## Boundaries and limitations

This notebook does not prove profitable FX arbitrage exists in live markets. It does not model fill probability, latency, counterparty risk, venue rules, financing, or order execution.

## Reproduction record

The notebook uses five deterministic offline quote snapshots and two directions per snapshot, producing ten opportunity rows.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "VERSION").is_file():
    if ROOT.parent == ROOT:
        raise RuntimeError("ZeroModel repository root not found")
    ROOT = ROOT.parent
os.chdir(ROOT)
for path in (ROOT, ROOT / "examples"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
print(f"repository root: {ROOT}")

In [ ]:
from examples.fx_triangular_arbitrage import demo_payload

payload = demo_payload()
selected = payload["selected"]

print("TOP-LEFT OPPORTUNITY")
print("Decision:", selected["decision"])
print("Cycle:", selected["cycle"])
print("Starting notional: EUR 100,000")
print("Expected profit: EUR", selected["expected_profit"])
print("Net edge bps:", selected["net_edge_bps"])
print("Quote skew ms:", selected["quote_skew_ms"])
print("Selected row:", selected["opportunity_id"])
print("Artifact ID:", payload["artifact_id"])
print()
print("Row order:")
for row_id in payload["row_order"]:
    print(" -", row_id)

ModuleNotFoundError: No module named 'examples'

## Why bid and ask matter

The forward cycle `EUR -> USD -> GBP -> EUR` uses:

```python
ending_eur = eurusd_bid / gbpusd_ask / eurgbp_ask
```

The reverse cycle `EUR -> GBP -> USD -> EUR` uses:

```python
ending_eur = eurgbp_bid * gbpusd_bid / eurusd_ask
```

Midpoints are not execution prices, so the example never uses midpoint prices for executable opportunity calculation.

In [ ]:
print("Source quotes for selected opportunity:")
for quote in selected["source_quotes"]:
    print(quote)
print("Arithmetic:", selected["arithmetic"])
print()
print("Stale profitable candidate:")
stale = payload["stale_rejection"]
print("Gross edge bps:", stale["gross_edge_bps"])
print("Net edge bps:", stale["net_edge_bps"])
print("Quote skew ms:", stale["quote_skew_ms"])
print("Decision:", stale["decision"])

## Mutation and replay

The mutation starts from the coherent snapshot, reduces exactly one `EUR/GBP` ask by 3 basis points, recompiles the VPM, and then restores the original quote. The decision and artifact identity change only while the quote is changed.

In [ ]:
print("Mutation:")
for key, value in payload["mutation"].items():
    print(f"{key}: {value}")
print()
print("Replay:")
for key, value in payload["replay"].items():
    print(f"{key}: {value}")